# Task 1: Exploratory Data Analysis and Preprocessing

**Objective**: Understand the CFPB complaint dataset, clean it, and prepare it for the RAG pipeline.

**Product Categories of Interest** (as per CrediTrust):
1. Credit Cards
2. Personal Loans
3. Buy Now Pay Later
4. Savings Accounts
5. Money Transfers

I will filter the dataset to these five categories and clean the complaint narratives.

# 1. Setting up the environment

In [ ]:
print("Setting Up and importing...")
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
%matplotlib inline

print("Libraries imported successfully!")

## 2. Load the Dataset Efficiently (Using Chunking)

The dataset is large. To avoid memory issues, we will load it in chunks using `pandas.read_csv` with `chunksize`. We'll process each chunk to:
- Map original product names to out target categories.
- Collect product distribution counts.
- Collect narrative length statistics.
- Filter and keep only rows matching our five product categories and having a narrative.

Finally, we'll combine the filtered rows into a single DataFrame and save it.

In [ ]:
product_mapping = {
    'Credit card': 'Credit card',
    'Credit card or prepaid card': 'Credit card',
    'Prepaid card': 'Credit card',
    'Personal loan': 'Personal loan',
    'Payday loan, title loan, or personal loan': 'Personal loan',
    'Payday loan, title loan, personal loan, or advance loan': 'Personal loan',
    'Payday loan': 'Personal loan',
    'Buy Now Pay Later': 'Buy Now Pay Later',                 # may be absent
    'Savings account': 'Savings account',
    'Checking or savings account': 'Savings account',         # treat as savings
    'Money transfer': 'Money transfer',
    'Money transfers': 'Money transfer',
    'Money transfer, virtual currency, or money service': 'Money transfer'
}

In [ ]:
# Define file path – adjust if your filename differs
file_path = '../data/raw/complaints.csv'

# Mapping from dataset product names to our five categories


target_categories = list(product_mapping.values())   # ['Credit card', 'Personal loan', 'Buy Now Pay Later', 'Savings account', 'Money transfer']

# Initialize accumulators
filtered_chunks = []          # will hold DataFrames of filtered rows
total_rows = 0
orig_product_counter = {}     # counts of original product names
mapped_product_counter = {}    # counts after mapping
narrative_word_counts = []     # word counts of narratives (for histogram)

# Choose chunk size based on your available memory (50k‑100k is safe)
chunk_size = 100000

print("Starting chunked processing...")
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)):
    print(f"  Processing chunk {i+1}...")
    total_rows += len(chunk)
    
    # 1. Count original product names (for overall distribution)
    orig_counts = chunk['Product'].value_counts().to_dict()
    for prod, cnt in orig_counts.items():
        orig_product_counter[prod] = orig_product_counter.get(prod, 0) + cnt
    
    # 2. Map to our categories
    chunk['mapped_product'] = chunk['Product'].map(product_mapping)
    
    # 3. Filter rows that map to a target category AND have a narrative
    mask = (chunk['mapped_product'].notna()) & (chunk['Consumer complaint narrative'].notna())
    filtered = chunk[mask].copy()
    
    if not filtered.empty:
        # Update mapped product counts
        mapped_counts = filtered['mapped_product'].value_counts().to_dict()
        for prod, cnt in mapped_counts.items():
            mapped_product_counter[prod] = mapped_product_counter.get(prod, 0) + cnt
        
        # Collect word counts for narrative length analysis
        word_counts = filtered['Consumer complaint narrative'].apply(lambda x: len(str(x).split()))
        narrative_word_counts.extend(word_counts.tolist())
        
        # Keep this chunk
        filtered_chunks.append(filtered)
    
    # Optional: stop after a few chunks for testing (remove when running on full data)
    # if i > 5: break

print("\nFinished processing all chunks.")

## 3. Overall product distribution (Original Dataset)

In [ ]:
# Convert original product counts to a Series and display top 20
orig_product_series = pd.Series(orig_product_counter).sort_values(ascending=False)
print("Top 20 product categories in the full dataset:")
print(orig_product_series.head(20))

## 4. Filtered data: After mapping and narrative requirement

In [ ]:
# Combine all filtered chunks into one DataFrame
if filtered_chunks:
    filtered_df = pd.concat(filtered_chunks, ignore_index=True)
    print(f"Total filtered rows (with narrative and mapped product): {len(filtered_df)}")
    print("\nDistribution by mapped product:")
    print(pd.Series(mapped_product_counter).sort_values(ascending=False))
    print("\nDistribution by original product (within filtered set):")
    print(filtered_df['Product'].value_counts())
else:
    filtered_df = pd.DataFrame()
    print("No rows matched the filter criteria.")

## 5. Narrative length analysis

In [ ]:
if narrative_word_counts:
    word_series = pd.Series(narrative_word_counts)
    print("Word count statistics (for filtered narratives):")
    print(word_series.describe())
    
    # Histogram (capped at 500 words for readability)
    plt.figure(figsize=(10,5))
    plt.hist(word_series.clip(upper=500), bins=50, edgecolor='black')
    plt.title('Distribution of Complaint Narrative Word Count (capped at 500)')
    plt.xlabel('Word Count')
    plt.ylabel('Frequency')
    plt.show()
    
    # Check extremes
    total_narr = len(word_series)
    short_pct = (word_series < 5).sum() / total_narr * 100
    long_pct = (word_series > 1000).sum() / total_narr * 100
    print(f"\nVery short narratives (<5 words): {short_pct:.2f}%")
    print(f"Very long narratives (>1000 words): {long_pct:.2f}%")
else:
    print("No narrative data available.")

## 6. Text cleaning
I clean the complaint narratives to improve embedding quality.  
Steps:
- Lowercase
- Remove special characters (keep letters, numbers, spaces)
- Remove extra whitespace

In [ ]:
def clean_text(text):
    """Basic text cleaning."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)   # keep alphanumeric and spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text
print("Cleaning text...")
# Apply cleaning
if not filtered_df.empty:
    filtered_df['cleaned_narrative'] = filtered_df['Consumer complaint narrative'].apply(clean_text)
    
    # Show example
    sample = filtered_df.sample(1, random_state=42)
    print("ORIGINAL NARRATIVE (first 500 chars):")
    print(sample['Consumer complaint narrative'].values[0][:500])
    print("\nCLEANED NARRATIVE (first 500 chars):")
    print(sample['cleaned_narrative'].values[0][:500])
else:
    print("No data to clean.")

## 7. Save Cleaned Dataset

We save the filtered and cleaned DataFrame to `data/processed/filtered_complaints.csv`.  
The file includes all original columns plus `mapped_product` and `cleaned_narrative`.

In [ ]:
if not filtered_df.empty:
    output_path = '../data/processed/filtered_complaints.csv'
    filtered_df.to_csv(output_path, index=False)
    print(f"Cleaned data saved to {output_path}")
    print(f"Shape of saved data: {filtered_df.shape}")
else:
    print("No data to save.")

## 8. Summary of key findings

The original CFPB dataset contained approximately 10 million complaints. After mapping to the five target products, approximately 464010 complaints remained. Of these, 464010 had non‑empty narratives and were kept for further processing.

The most common product category was Credit Cards, followed by Personal Loans. "Buy Now Pay Later" did not appear in the historical data, as expected.

Narrative lengths varied widely, with a median of around 150 words. Very short narratives (<5 words): 0.06%
Very long narratives (>1000 words): 1.17%
Overall, the data appears reasonably clean, with no major encoding issues.